In [16]:
pip install pulp


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import numpy as np
from scipy.optimize import linprog

In [18]:
def predict_delay(distance, weather_score, traffic_score):
    distance_factor = distance / 1000
    weather_factor = weather_score / 100
    traffic_factor = traffic_score / 100

    delay_score = (
        0.5 * distance_factor +
        0.3 * weather_factor +
        0.2 * traffic_factor
    )

    return 1 if delay_score > 0.5 else 0

In [19]:
def optimize_strategy(costs, time_effect, budget, required_reduction):
    import pulp

    prob = pulp.LpProblem("SupplyChainOptimization", pulp.LpMinimize)

    x = [pulp.LpVariable(f"x{i}", cat='Binary') for i in range(len(costs))]

    # Minimize cost
    prob += pulp.lpSum([costs[i] * x[i] for i in range(len(costs))])

    # Budget constraint
    prob += pulp.lpSum([costs[i] * x[i] for i in range(len(costs))]) <= budget

    # FIXED: Must achieve required delay reduction
    prob += pulp.lpSum([time_effect[i] * x[i] for i in range(len(costs))]) >= required_reduction

    prob.solve()

    selected = [i for i in range(len(x)) if x[i].value() == 1]

    total_cost = sum(costs[i] for i in selected)
    total_time = sum(time_effect[i] for i in selected)

    return selected, total_cost, total_time

In [20]:
def get_recommendations():
    actions = ["Air Shipping", "Extra Inventory", "New Supplier"]

    costs = [800, 300, 500]
    time_effect = [2, 5, 3]

    solutions = []

    res1 = optimize_strategy(costs, time_effect, 1500, 10)
    solutions.append(("Low Cost", costs, res1))

    res2 = optimize_strategy(costs, time_effect, 3000, 4)
    solutions.append(("Fast Delivery", costs, res2))

    res3 = optimize_strategy(costs, time_effect, 2000, 6)
    solutions.append(("Balanced", costs, res3))

    return actions, time_effect, solutions

In [21]:
distance = 400
weather = 60
traffic = 70

# Step 1: Predict delay
delay = predict_delay(distance, weather, traffic)

# Step 2: If delay → run optimization
if delay:
    print("⚠️ Delay Predicted!\n")

    actions, time_effect, solutions = get_recommendations()

    # Direct printing (safe version)
    for name, costs, res in solutions:
        selected, total_cost, total_time = res

        chosen_actions = [actions[i] for i in selected]

        print("\n==============================")
        print(f"Strategy: {name}")
        print("==============================")
        print("Actions:", chosen_actions if chosen_actions else "None")
        print(f"Total Cost: ${total_cost:.2f}")
        print(f"Time Impact: {total_time:.2f} days")

        if name == "Low Cost":
            print("Trade-off: Cheaper but slower")
        elif name == "Fast Delivery":
            print("Trade-off: Faster but expensive")
        else:
            print("Trade-off: Balanced cost and speed")

# Step 3: No delay case
else:
    print("✅ No Delay Predicted")

⚠️ Delay Predicted!


Strategy: Low Cost
Actions: ['Air Shipping', 'Extra Inventory', 'New Supplier']
Total Cost: $1600.00
Time Impact: 10.00 days
Trade-off: Cheaper but slower

Strategy: Fast Delivery
Actions: ['Extra Inventory']
Total Cost: $300.00
Time Impact: 5.00 days
Trade-off: Faster but expensive

Strategy: Balanced
Actions: ['Extra Inventory', 'New Supplier']
Total Cost: $800.00
Time Impact: 8.00 days
Trade-off: Balanced cost and speed
